# B2 – Llama-2 7B vs 13B as DDX engine (Hugging Face Spaces API)

The Spaces are ZeroGPU: free accounts get a few GPU-minutes per day, so **log in with `HF_TOKEN`** (more quota than anonymous). If you hit *GPU quota exceeded*, either wait for the reset or use the Ollama fallback at the bottom. The Space API is stateless, so this notebook uses `mode="single"`.

In [1]:
import sys, os, json, time, gc
sys.path.insert(0, "src")
from dotenv import load_dotenv; load_dotenv()
import pandas as pd
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 200)
from vignettes import load_vignettes, sample_vignettes
from protocol import run_case, SYSTEM_DOCTOR
import clients, store, metrics

In [2]:
from pathlib import Path

# Move up from `notebook/` to project root once, even if re-run
if Path.cwd().name == "notebooks":
  os.chdir("..")

In [3]:
VIGNETTES = load_vignettes("data/Clinical_Vignettes_26_27.txt")
print(len(VIGNETTES), "clean 4-line vignettes")

117 clean 4-line vignettes


In [12]:
SPACES = {"llama2-7b": "huggingface-projects/llama-2-7b-chat",
          "llama2-13b": "huggingface-projects/llama-2-13b-chat"}
bots = {}
for name, sid in SPACES.items():
    try:
        bots[name] = clients.make_hf_space(sid, api_name="/generate", max_new_tokens=512, temperature=0.1)
        print("=====", name); bots[name].client.view_api()
    except Exception as e:
        print(name, "-> unavailable:", repr(e)[:300])

Loaded as API: https://huggingface-projects-llama-2-7b-chat.hf.space
===== llama2-7b
Client.predict() Usage Info
---------------------------
Named API endpoints: 1

 - predict(message, system_prompt, max_new_tokens, temperature, top_p, top_k, repetition_penalty, api_name="/generate") -> response
    Parameters:
     - [Textbox] message: str (required)  
     - [Textbox] system_prompt: str (required)  
     - [Slider] max_new_tokens: float (not required, defaults to:   1024)  
     - [Slider] temperature: float (not required, defaults to:   0.6)  
     - [Slider] top_p: float (not required, defaults to:   0.9)  
     - [Slider] top_k: float (not required, defaults to:   50)  
     - [Slider] repetition_penalty: float (not required, defaults to:   1.2)  
    Returns:
     - [Json] response: str | float | bool | list | dict (any valid json) 

Loaded as API: https://huggingface-projects-llama-2-13b-chat.hf.space
===== llama2-13b
Client.predict() Usage Info
---------------------------
Named

Read `view_api()` above. If the endpoint name or parameter names differ from `/chat` + `message`/`system_prompt`, adapt `make_hf_space(...)` (extra settings go in `**params`, e.g. `max_new_tokens=512, temperature=0.2`).

In [5]:
# identical call to notebook 03 (n_core=5, n_extra=25, seed=2026) so the 5 core vignettes are the SAME for every bot
SAMPLE = [v for v in sample_vignettes(VIGNETTES, 5, 25, 2026) if v["set"] == "core"]
print([v["id"] for v in SAMPLE])
done = store.done_keys()
for name, fn in bots.items():
    for v in SAMPLE:
        if (name, v["id"], "single") in done: continue
        try:
            res = run_case(fn, v, mode="single")
        except Exception as e:
            print("STOP", name, v["id"], repr(e)[:300]); break      # usually quota exceeded
        store.save_run(name, res); print(name, v["id"], "ok")

[15, 40, 64, 65, 82]
llama2-7b 15 ok
llama2-7b 40 ok
llama2-7b 64 ok
llama2-7b 65 ok
llama2-7b 82 ok
llama2-13b 15 ok
llama2-13b 40 ok
llama2-13b 64 ok
STOP llama2-13b 65 AppError('You have exceeded your free ZeroGPU quota (90s requested vs. 0s left). Try again in 0:00:00. Subscribe to Hugging Face PRO to get 40 min of ZeroGPU quota a day - https://huggingface.co/subscribe/pro?from=ZeroGPU')


## Fallback: local Llama-2 through Ollama (CPU, runs on the iMac)
On the Mac (not in Docker): `brew install ollama && ollama serve`, then `ollama pull llama2:7b-chat` and `ollama pull llama2:13b-chat` (check exact tags with `ollama list` / the Ollama library). 13B at 4-bit needs ~8 GB RAM; expect slow CPU generation. State in the report which route you used.

In [4]:
import os, requests
host = os.environ.get("OLLAMA_HOST", "http://host.docker.internal:11434")
print("host:", host)
print([m["name"] for m in requests.get(f"{host}/api/tags", timeout=30).json().get("models", [])])
r = requests.post(f"{host}/api/chat", json={
    "model": "llama2:7b-chat", "stream": False,
    "messages": [{"role": "user", "content": "Say hi"}]}, timeout=900)
print(r.status_code, r.text[:600])

host: http://host.docker.internal:11434
['llama2:13b-chat', 'llama2:7b-chat']
200 {"model":"llama2:7b-chat","created_at":"2026-11-26T16:38:15.578541Z","message":{"role":"assistant","content":"Hi there! *waves* How are you today?"},"done":true,"done_reason":"stop","total_duration":7375459747,"load_duration":4799637017,"prompt_eval_count":23,"prompt_eval_cached_count":0,"prompt_eval_duration":683177000,"eval_count":13,"eval_duration":1890975000}


In [6]:
# identical call to notebook 03 (n_core=5, n_extra=25, seed=2026) so the 5 core vignettes are the SAME for every bot
SAMPLE = [v for v in sample_vignettes(VIGNETTES, 5, 25, 2026) if v["set"] == "core"]
print([v["id"] for v in SAMPLE])
done = store.done_keys()

[15, 40, 64, 65, 82]


In [9]:
USE_OLLAMA = True
if USE_OLLAMA:
    for name, tag in {"llama2-7b-ollama": "llama2:7b-chat", "llama2-13b-ollama": "llama2:13b-chat"}.items():
        fn = clients.make_ollama(tag, temperature=0.0)
        for v in SAMPLE:
            if (name, v["id"], "single") in store.done_keys(): continue
            store.save_run(name, run_case(fn, v, mode="single")); print(name, v["id"], "ok")

## Manual route (what the practice literally asks)
For each of the 5 ids above, paste `p1 + p2` into the Space UI, copy the reply, and compare with the `gold` line:

In [8]:
for v in SAMPLE:
    print(f"--- vignette {v['id']} ---\nPATIENT: {v['p1']} {v['p2']}\nGOLD:    {v['gold']}\n")

--- vignette 15 ---
PATIENT: Doctor, I've been feeling extremely anxious and have panic attacks. Crowded places and public speaking make me anxious.
GOLD:    It sounds like you might have social anxiety disorder. We can explore treatment options.

--- vignette 40 ---
PATIENT: Doctor, I've had sharp abdominal pain on my left side. My bowel movements have changed, and I feel bloated.
GOLD:    Your symptoms raise concerns about diverticulitis. We'll perform imaging tests.

--- vignette 64 ---
PATIENT: Doctor, I've been waking up with severe headaches and nausea. Yes, they mainly happen in the morning when I wake up.
GOLD:    Your symptoms may be linked to increased intracranial pressure. We'll order imaging to investigate further.

--- vignette 65 ---
PATIENT: Doctor, I've been having frequent muscle spasms and cramps. I've been trying, but the spasms persist even when I'm hydrated.
GOLD:    Your symptoms might be due to an electrolyte imbalance. We'll check your blood levels.

--- vignet

## Check the results

In [11]:
runs = pd.DataFrame(store.load_runs())
l2 = runs[runs.bot.str.startswith("llama2")].sort_values(["vignette_id", "bot"])
for _, r in l2.iterrows():
    print(f"[{r.bot} | vignette {r.vignette_id}] {r.latency_s:.0f}s\n{r.final_text}\n" + "-"*60)

[llama2-13b | vignette 15] 11s
Certainly! Based on what you've shared, here is my ranked list of the top 3 most likely diagnoses for your symptoms:

1. Generalized Anxiety Disorder (GAD): Your persistent and excessive worry about everyday things, along with your avoidance behaviors in crowded places and public speaking, point towards GAD as a strong possibility.
2. Panic Disorder: The frequency and intensity of your panic attacks suggest that this could be a contributing factor to your anxiety.
3. Social Anxiety Disorder (SAD): Your fear of being judged or evaluated negatively by others in social situations aligns well with SAD.

Most likely diagnosis: Generalized Anxiety Disorder (GAD).
------------------------------------------------------------
[llama2-13b-ollama | vignette 15] 99s
As an experienced physician, based on the information you have provided, my top 3 most likely diagnoses for your symptoms of anxiety and panic attacks are:

1. Generalized Anxiety Disorder (GAD): GAD is a

## Print the answers

In [12]:
ol = runs[runs.bot.str.endswith("llama2-13b")].sort_values(["vignette_id", "bot"])
for _, r in ol.iterrows():
    print(f"[{r.bot} | vignette {r.vignette_id}] {r.latency_s:.0f}s\n{r.final_text}\n" + "-"*60)

[llama2-13b | vignette 15] 11s
Certainly! Based on what you've shared, here is my ranked list of the top 3 most likely diagnoses for your symptoms:

1. Generalized Anxiety Disorder (GAD): Your persistent and excessive worry about everyday things, along with your avoidance behaviors in crowded places and public speaking, point towards GAD as a strong possibility.
2. Panic Disorder: The frequency and intensity of your panic attacks suggest that this could be a contributing factor to your anxiety.
3. Social Anxiety Disorder (SAD): Your fear of being judged or evaluated negatively by others in social situations aligns well with SAD.

Most likely diagnosis: Generalized Anxiety Disorder (GAD).
------------------------------------------------------------
[llama2-13b | vignette 40] 7s
Sure thing! Here's my ranked list of the top 3 most likely diagnoses based on what you've shared:

1. Appendicitis: Your sharp abdominal pain on the left side, especially given its sudden onset, suggests appendic